In [1]:
# ====================== 1. 安装依赖（首次运行） ======================
!pip install torch jieba numpy

# ====================== 2. 导入所有库 ======================
import torch
import torch.nn as nn
import jieba
import numpy as np
from torch.utils.data import DataLoader, TensorDataset

# ====================== 3. 全局配置 ======================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EMBEDDING_SIZE = 128
HIDDEN_SIZE = 256
NUM_CLASSES = 4    # 分类类别数
DROPOUT = 0.5
MAX_LEN = 20       # 句子最大长度
BATCH_SIZE = 8
EPOCHS = 10
LR = 0.001

# ====================== 4. RNN 模型定义 ======================
class RNN(nn.Module):
    def __init__(self, word_count, embedding_size, hidden_size, output_size):
        super(RNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(word_count, embedding_size)
        self.i2h = nn.Linear(embedding_size + hidden_size, hidden_size)
        self.i2o = nn.Linear(embedding_size + hidden_size, output_size)
        self.dropout = nn.Dropout(DROPOUT)
        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.embedding.weight)
        nn.init.xavier_uniform_(self.i2h.weight)
        nn.init.xavier_uniform_(self.i2o.weight)
        nn.init.zeros_(self.i2h.bias)
        nn.init.zeros_(self.i2o.bias)

    def forward(self, input_tensor, hidden):
        word_vector = self.embedding(input_tensor)
        combined = torch.cat((word_vector, hidden), dim=1)
        hidden = self.i2h(combined)
        hidden = self.dropout(hidden)
        output = self.i2o(combined)
        return output, hidden

    def initHidden(self):
        return torch.zeros(1, self.hidden_size).to(DEVICE)

# ====================== 5. 工具函数 ======================
def tokenize(text):
    return list(jieba.cut(text.strip()))

def build_vocab(sentences):
    vocab = {"<PAD>": 0}
    for sen in sentences:
        for word in sen:
            if word not in vocab:
                vocab[word] = len(vocab)
    return vocab

def sen2idx(sen, vocab, max_len):
    idx = [vocab.get(word, 0) for word in sen]
    if len(idx) < max_len:
        idx += [0] * (max_len - len(idx))
    return idx[:max_len]

def get_category(model, text, vocab, max_len=MAX_LEN):
    model.eval()
    words = tokenize(text)
    idx = sen2idx(words, vocab, max_len)
    tensor = torch.tensor(idx, dtype=torch.long).to(DEVICE)
    
    hidden = model.initHidden()
    for i in range(tensor.size(0)):
        output, hidden = model(tensor[i].unsqueeze(0), hidden)
    
    pred = torch.argmax(output, dim=1).item()
    return pred

# ====================== 6. 构造示例数据集（可自己替换） ======================
# 文本 + 标签（0:招聘 1:考研 2:学校 3:导师）
texts = [
    "校招今日头条后端开发工程师",
    "腾讯算法岗招聘",
    "考上清华大学计算机研究生",
    "考研院校怎么选",
    "北京大学介绍",
    "复旦大学专业排名",
    "导师组研究方向",
    "导师联系方式"
]
labels = [0, 0, 1, 1, 2, 2, 3, 3]

# 分词 + 构建词典
tokenized = [tokenize(t) for t in texts]
vocab = build_vocab(tokenized)
VOCAB_SIZE = len(vocab)

# 转索引
features = [sen2idx(s, vocab, MAX_LEN) for s in tokenized]
x = torch.tensor(features, dtype=torch.long)
y = torch.tensor(labels, dtype=torch.long)

# 数据加载器
dataset = TensorDataset(x, y)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# ====================== 7. 模型训练 ======================
model = RNN(VOCAB_SIZE, EMBEDDING_SIZE, HIDDEN_SIZE, NUM_CLASSES).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

print("开始训练...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch_x, batch_y in loader:
        batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
        
        loss = 0
        for seq in range(batch_x.size(0)):
            hidden = model.initHidden()
            for i in range(MAX_LEN):
                output, hidden = model(batch_x[seq][i].unsqueeze(0), hidden)
            loss += criterion(output, batch_y[seq].unsqueeze(0))
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1:2d} | Loss: {total_loss:.4f}")

# ====================== 8. 测试模型 ======================
print("\n" + "="*50)
print("模型测试结果")
print("="*50)

test_list = [
    "校招今日头条后端开发工程师",
    "你考上清华计算机研究生了",
    "北京大学怎么样",
    "导师组介绍"
]

id2label = {0:"招聘", 1:"考研", 2:"学校", 3:"导师"}
for t in test_list:
    pred = get_category(model, t, vocab)
    print(f"输入：{t}")
    print(f"预测类别：{id2label[pred]}\n")


[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\叶湘伦\AppData\Local\Temp\jieba.cache
Loading model cost 0.416 seconds.
Prefix dict has been built successfully.


开始训练...
Epoch  1 | Loss: 148.6943
Epoch  2 | Loss: 99.6633
Epoch  3 | Loss: 150.5021
Epoch  4 | Loss: 63.9851
Epoch  5 | Loss: 145.9040
Epoch  6 | Loss: 76.3489
Epoch  7 | Loss: 140.3177
Epoch  8 | Loss: 127.7745
Epoch  9 | Loss: 123.9909
Epoch 10 | Loss: 124.5518

模型测试结果
输入：校招今日头条后端开发工程师
预测类别：考研

输入：你考上清华计算机研究生了
预测类别：招聘

输入：北京大学怎么样
预测类别：考研

输入：导师组介绍
预测类别：考研



In [3]:
# ====================== 1. 安装依赖 ======================
!pip install torch jieba numpy

# ====================== 2. 导入库 ======================
import torch
import torch.nn as nn
import torch.optim as optim
import jieba
from torch.utils.data import DataLoader, TensorDataset

# ====================== 3. 全局配置 ======================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EMBEDDING_SIZE = 128
HIDDEN_SIZE = 256
NUM_CLASSES = 4    # 0:招聘 1:考研 2:学校 3:导师
DROPOUT = 0.5
MAX_LEN = 20
BATCH_SIZE = 4
EPOCHS = 50
LR = 0.001

# ====================== 4. 标准 RNN 文本分类模型 ======================
class TextRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True, bidirectional=False)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        # x: [batch, seq_len]
        embed = self.embedding(x)  # [batch, seq_len, embed_dim]
        out, hidden = self.rnn(embed)  # out: [batch, seq_len, hidden_dim]
        last_hidden = hidden[-1, :, :]  # [batch, hidden_dim]
        out = self.dropout(last_hidden)
        logits = self.fc(out)  # [batch, num_classes]
        return logits

# ====================== 5. 工具函数 ======================
def tokenize(text):
    return list(jieba.cut(text.strip()))

def build_vocab(sentences):
    vocab = {"<PAD>": 0, "<UNK>": 1}
    for sen in sentences:
        for word in sen:
            if word not in vocab:
                vocab[word] = len(vocab)
    return vocab

def sen2idx(sen, vocab, max_len):
    idx = [vocab.get(word, vocab["<UNK>"]) for word in sen]
    if len(idx) < max_len:
        idx += [vocab["<PAD>"]] * (max_len - len(idx))
    return idx[:max_len]

def predict_text(model, text, vocab, max_len=MAX_LEN):
    model.eval()
    words = tokenize(text)
    idx = sen2idx(words, vocab, max_len)
    tensor = torch.tensor([idx], dtype=torch.long).to(DEVICE)
    with torch.no_grad():
        logits = model(tensor)
        pred = torch.argmax(logits, dim=1).item()
    return pred

# ====================== 6. 扩充后的数据集 ======================
texts = [
    # 0:招聘
    "校招今日头条后端开发工程师",
    "腾讯算法岗招聘",
    "字节跳动春招岗位",
    "阿里巴巴校招流程",
    "互联网大厂秋招经验",
    "前端开发岗位招聘",
    "后端开发工程师校招",
    "校招面试经验分享",
    # 1:考研
    "考上清华大学计算机研究生",
    "考研院校怎么选",
    "计算机考研复习计划",
    "考研复试经验分享",
    "考研数学怎么学",
    "考研英语复习方法",
    "跨专业考研经验",
    "计算机考研院校推荐",
    # 2:学校
    "北京大学怎么样",
    "复旦大学专业排名",
    "清华大学介绍",
    "上海交通大学王牌专业",
    "985院校名单",
    "211大学推荐",
    "计算机专业强校有哪些",
    "高校专业排名",
    # 3:导师
    "导师组研究方向",
    "导师联系方式",
    "如何联系导师",
    "导师组介绍",
    "选导师注意事项",
    "导师研究方向介绍",
    "怎么和导师沟通",
    "导师组构成"
]
labels = [0]*8 + [1]*8 + [2]*8 + [3]*8

# 分词 + 构建词典
tokenized = [tokenize(t) for t in texts]
vocab = build_vocab(tokenized)
VOCAB_SIZE = len(vocab)

# 转索引
features = [sen2idx(s, vocab, MAX_LEN) for s in tokenized]
x = torch.tensor(features, dtype=torch.long)
y = torch.tensor(labels, dtype=torch.long)

# 数据加载器
dataset = TensorDataset(x, y)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# ====================== 7. 训练 ======================
model = TextRNN(VOCAB_SIZE, EMBEDDING_SIZE, HIDDEN_SIZE, NUM_CLASSES, DROPOUT).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

print("开始训练...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch_x, batch_y in loader:
        batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
        
        optimizer.zero_grad()
        logits = model(batch_x)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * batch_x.size(0)
    
    avg_loss = total_loss / len(dataset)
    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1:2d} | Avg Loss: {avg_loss:.4f}")

# ====================== 8. 测试 ======================
print("\n" + "="*50)
print("模型测试结果")
print("="*50)

test_list = [
    "校招今日头条后端开发工程师",
    "你考上清华计算机研究生了",
    "北京大学怎么样",
    "导师组介绍"
]

id2label = {0:"招聘", 1:"考研", 2:"学校", 3:"导师"}
for t in test_list:
    pred = predict_text(model, t, vocab)
    print(f"输入：{t}")
    print(f"预测类别：{id2label[pred]}\n")


[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


开始训练...
Epoch  5 | Avg Loss: 1.3910
Epoch 10 | Avg Loss: 1.1946
Epoch 15 | Avg Loss: 0.5372
Epoch 20 | Avg Loss: 0.2507
Epoch 25 | Avg Loss: 0.1354
Epoch 30 | Avg Loss: 0.4435
Epoch 35 | Avg Loss: 0.3873
Epoch 40 | Avg Loss: 0.6767
Epoch 45 | Avg Loss: 0.4455
Epoch 50 | Avg Loss: 0.3996

模型测试结果
输入：校招今日头条后端开发工程师
预测类别：招聘

输入：你考上清华计算机研究生了
预测类别：招聘

输入：北京大学怎么样
预测类别：学校

输入：导师组介绍
预测类别：导师



In [2]:
# ====================== 1. 安装依赖 ======================
!pip install torch jieba numpy

# ====================== 2. 导入库 ======================
import torch
import torch.nn as nn
import torch.optim as optim
import jieba
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

# ====================== 3. 全局配置 ======================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EMBEDDING_SIZE = 128
HIDDEN_SIZE = 256
NUM_CLASSES = 4    # 0:招聘 1:考研 2:学校 3:导师
DROPOUT = 0.5
MAX_LEN = 20
BATCH_SIZE = 4
EPOCHS = 50
LR = 0.001

# ====================== 4. 标准 RNN 文本分类模型 ======================
class TextRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True, bidirectional=False)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        embed = self.embedding(x)
        out, hidden = self.rnn(embed)
        last_hidden = hidden[-1, :, :]
        out = self.dropout(last_hidden)
        logits = self.fc(out)
        return logits

# ====================== 5. 工具函数 ======================
def tokenize(text):
    return list(jieba.cut(text.strip()))

def build_vocab(sentences):
    vocab = {"<PAD>": 0, "<UNK>": 1}
    for sen in sentences:
        for word in sen:
            if word not in vocab:
                vocab[word] = len(vocab)
    return vocab

def sen2idx(sen, vocab, max_len):
    idx = [vocab.get(word, vocab["<UNK>"]) for word in sen]
    if len(idx) < max_len:
        idx += [vocab["<PAD>"]] * (max_len - len(idx))
    return idx[:max_len]

def predict_text(model, text, vocab, max_len=MAX_LEN):
    model.eval()
    words = tokenize(text)
    idx = sen2idx(words, vocab, max_len)
    tensor = torch.tensor([idx], dtype=torch.long).to(DEVICE)
    with torch.no_grad():
        logits = model(tensor)
        pred = torch.argmax(logits, dim=1).item()
    return pred

# ====================== 6. 扩充后的数据集 ======================
texts = [
    # 0:招聘
    "校招今日头条后端开发工程师",
    "腾讯算法岗招聘",
    "字节跳动春招岗位",
    "阿里巴巴校招流程",
    "互联网大厂秋招经验",
    "前端开发岗位招聘",
    "后端开发工程师校招",
    "校招面试经验分享",
    # 1:考研
    "考上清华大学计算机研究生",
    "考研院校怎么选",
    "计算机考研复习计划",
    "考研复试经验分享",
    "考研数学怎么学",
    "考研英语复习方法",
    "跨专业考研经验",
    "计算机考研院校推荐",
    # 2:学校
    "北京大学怎么样",
    "复旦大学专业排名",
    "清华大学介绍",
    "上海交通大学王牌专业",
    "985院校名单",
    "211大学推荐",
    "计算机专业强校有哪些",
    "高校专业排名",
    # 3:导师
    "导师组研究方向",
    "导师联系方式",
    "如何联系导师",
    "导师组介绍",
    "选导师注意事项",
    "导师研究方向介绍",
    "怎么和导师沟通",
    "导师组构成"
]
labels = [0]*8 + [1]*8 + [2]*8 + [3]*8

# 分词 + 构建词典
tokenized = [tokenize(t) for t in texts]
vocab = build_vocab(tokenized)
VOCAB_SIZE = len(vocab)

# 转索引
features = [sen2idx(s, vocab, MAX_LEN) for s in tokenized]
x = torch.tensor(features, dtype=torch.long)
y = torch.tensor(labels, dtype=torch.long)

# 数据加载器
dataset = TensorDataset(x, y)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# ====================== 7. 训练（含准确率） ======================
model = TextRNN(VOCAB_SIZE, EMBEDDING_SIZE, HIDDEN_SIZE, NUM_CLASSES, DROPOUT).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

print("开始训练...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch_x, batch_y in loader:
        batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
        
        optimizer.zero_grad()
        logits = model(batch_x)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * batch_x.size(0)
        
        # 用 PyTorch 原生方法计算准确率
        preds = torch.argmax(logits, dim=1)
        correct += (preds == batch_y).sum().item()
        total += batch_y.size(0)
    
    avg_loss = total_loss / len(dataset)
    acc = correct / total
    
    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1:2d} | Loss: {avg_loss:.4f} | Acc: {acc:.4f}")

# ====================== 8. 测试 ======================
print("\n" + "="*50)
print("模型测试结果")
print("="*50)

test_list = [
    "校招今日头条后端开发工程师",
    "你考上清华计算机研究生了",
    "北京大学怎么样",
    "导师组介绍"
]

id2label = {0:"招聘", 1:"考研", 2:"学校", 3:"导师"}
for t in test_list:
    pred = predict_text(model, t, vocab)
    print(f"输入：{t}")
    print(f"预测类别：{id2label[pred]}\n")


[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\叶湘伦\AppData\Local\Temp\jieba.cache


Loading model cost 0.380 seconds.
Prefix dict has been built successfully.


开始训练...
Epoch  5 | Loss: 1.3300 | Acc: 0.1875
Epoch 10 | Loss: 1.0476 | Acc: 0.4062
Epoch 15 | Loss: 0.9237 | Acc: 0.4688
Epoch 20 | Loss: 0.5416 | Acc: 0.7188
Epoch 25 | Loss: 0.1958 | Acc: 0.9688
Epoch 30 | Loss: 0.0206 | Acc: 1.0000
Epoch 35 | Loss: 0.0171 | Acc: 1.0000
Epoch 40 | Loss: 0.0104 | Acc: 1.0000
Epoch 45 | Loss: 0.0094 | Acc: 1.0000
Epoch 50 | Loss: 0.0070 | Acc: 1.0000

模型测试结果
输入：校招今日头条后端开发工程师
预测类别：招聘

输入：你考上清华计算机研究生了
预测类别：招聘

输入：北京大学怎么样
预测类别：学校

输入：导师组介绍
预测类别：导师

